In [2]:

# Algorithms for DNA Sequencing
# Johns Hopkins University
# modulo 2 

# 1 - 799954 
# 2 - 984143
# 3 - 127974
# 4 - 19
# 5 - 90
# 6 - 35

In [3]:

from Bio import SeqIO


fasta_file = "/home/pedroaraujo/FASTA FILE/chr1.GRCh38.excerpt.fasta"


for record in SeqIO.parse(fasta_file, "fasta"):
    print(record.id)       # cabeçalho
    print(record.seq[:40])      # sequência



CM000663.2_excerpt
TTGAATGCTGAAATCAGCAGGTAATATATGATAATAGAGA


In [4]:
# 1 



def naive_with_counts(p, t):
    n = len(t)
    m = len(p)
    alignments = 0 
    comparisons = 0
    positions = []

    for i in range(n - m + 1):
        alignments += 1
        match = True
        for j in range(m):
            comparisons += 1
            if t[i + j] != p[j]:
                match = False
                break
        if match:
            positions.append(i)
    return positions, comparisons, alignments




In [5]:
#1 e 2  
p = "GGCGCGGTGGCTCACGCCTGTAATCCCAGCACTTTGGGAGGCCGAGG"

t = str(record.seq)
print(naive_with_counts(p, t))

start = 56922
end = start + len(p)
print(p)
print(len(t) - len(p) +1)

([56922], 984143, 799954)
GGCGCGGTGGCTCACGCCTGTAATCCCAGCACTTTGGGAGGCCGAGG
799954


In [6]:
num_aligments = len(t) - len(p) + 1 
print(num_aligments)

799954


In [7]:
# 3
def boyer_moore_with_counts(p, p_bm, t):
    i = 0
    occurrences = []
    num_alignments = 0
    num_character_comparisons = 0

    while i <= len(t) - len(p):
        num_alignments += 1
        shift = 1
        mismatched = False

        for j in range(len(p) - 1, -1, -1):
            num_character_comparisons += 1
            if p[j] != t[i + j]:
                bc = p_bm.bad_character_rule(j, t[i + j])
                gs = p_bm.good_suffix_rule(j)
                shift = max(shift, bc, gs)
                mismatched = True
                break

        if not mismatched:
            occurrences.append(i)
            shift = p_bm.match_skip()

        i += shift

    return occurrences, num_alignments, num_character_comparisons






In [8]:
from bm_preproc import BoyerMoore
p = "GGCGCGGTGGCTCACGCCTGTAATCCCAGCACTTTGGGAGGCCGAGG"
t = record.seq


p_bm = BoyerMoore(p)

print(boyer_moore_with_counts(p, p_bm, t))





([56922], 127974, 165191)


In [9]:
# 4 

def approximate_match(p, t, n):
    segment_length = int(round(len(p) / (n + 1)))
    all_matches = set()

    for i in range(n + 1):
        start = i * segment_length
        end = min((i + 1) * segment_length, len(p))

        p_bm = BoyerMoore(p[start:end])
        matches, _, _ = boyer_moore_with_counts(p[start:end], p_bm, t)

        for m in matches:
            # Checa se o padrão completo cabe no texto
            if m < start or m - start + len(p) > len(t):
                continue

            mismatches = 0  # ✅ sempre inicializada aqui

            # Parte antes do segmento
            for j in range(0, start):
                if p[j] != t[m - start + j]:
                    mismatches += 1
                    if mismatches > n:
                        break

            # Parte depois do segmento
            if mismatches <= n:
                for j in range(end, len(p)):
                    if p[j] != t[m - start + j]:
                        mismatches += 1
                        if mismatches > n:
                            break

            if mismatches <= n:
                all_matches.add(m - start)

    return list(all_matches)









In [10]:
p = "GGCGCGGTGGCTCACGCCTGTAAT"
t = record.seq

print(approximate_match(p, t, 2))
print(len(approximate_match(p, t, 2)))



[273669, 681737, 717706, 262042, 635931, 84641, 160162, 724927, 657496, 160729, 56922, 191452, 551134, 747359, 421221, 147558, 364263, 465647, 429299]
19


In [11]:
 def query_subseq(p, t, subseq_ind, max_mismatches=2):
        occurrences = []
        num_index_hits = 0

        hits = subseq_ind.query(p)
        num_index_hits = len(hits)

        for i in hits:
            if i < 0 or i + len(p) > len(t):
                continue

            mismatches = 0
            for j in range(len(p)):
                if t[i + j] != p[j]:
                    mismatches += 1
                    if mismatches > max_mismatches:
                        break

            if mismatches <= max_mismatches:
               occurrences.append(i)

        return occurrences, num_index_hits

In [12]:
# 5 
from kmer_index import Index


p = "GGCGCGGTGGCTCACGCCTGTAAT"
t = str(record.seq)   # MUITO IMPORTANTE
d = 2

num_parts = d + 1
part_length = len(p) // num_parts  # 8

# ======= criar partes =======
parts = []
for i in range(num_parts):
    start = i * part_length
    end = start + part_length
    parts.append(p[start:end])

# ======= criar índice =======
k = part_length
index = Index(t, k)

# ======= contar index hits =======
total_hits = 0
for part in parts:
    hits = index.query(part)
    total_hits += len(hits)

print("Total index hits:", total_hits)


Total index hits: 90


In [13]:
from Bio import SeqIO

DNA = record.seq

template_strand = DNA.complement()

print(DNA[:40])
print(template_strand[:40])

RNA = DNA.transcribe()
print(RNA[:40])

translate = RNA.translate()
print(translate[:40])


GC_content = (DNA.count("G") + DNA.count("C")) *100/ len(DNA)

print(GC_content)




TTGAATGCTGAAATCAGCAGGTAATATATGATAATAGAGA
AACTTACGACTTTAGTCGTCCATTATATACTATTATCTCT
UUGAAUGCUGAAAUCAGCAGGUAAUAUAUGAUAAUAGAGA
LNAEISR*YMIIEKAIPKVHRSTILEPNSVDPKRKQFLLL
35.759375


/home/pedroaraujo/bioenv/lib/python3.10/site-packages/Bio/Seq.py:2877: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an error in future.
  warnings.warn(


In [14]:
seq = str(record.seq[:10])

for base in seq:
    print(base) 

T
T
G
A
A
T
G
C
T
G


In [15]:
seq = record.seq[:10]
i = 0

while i < len(seq):
    print(i,seq[i])
    i += 1 

0 T
1 T
2 G
3 A
4 A
5 T
6 G
7 C
8 T
9 G


In [16]:
def gc_content(DNA):
    gc = 0
    for base in seq:
        if base == "G" or base == "C":
            gc += 1
    return gc*100 / len(seq)

print(gc_content(DNA))

   









40.0


In [17]:
import importlib
import kmer_index
importlib.reload(kmer_index)
dir(kmer_index)



['Index',
 'SubseqIndex',
 '__author__',
 '__builtins__',
 '__cached__',
 '__doc__',
 '__file__',
 '__loader__',
 '__name__',
 '__package__',
 '__spec__',
 'bisect',
 'query_subseq']

In [18]:
 def query_subseq(p, t, subseq_ind, max_mismatches = 2):
        occurrences = []
        num_index_hits = 0
        hits = subseq_ind.query(p)
        num_index_hits = len(hits)

        for i in hits:
            if i < 0 or i + len(p) > len(t):
                continue

            mismatches = 0
            for j in range(len(p)):
                if t[i + j] != p[j]:
                    mismatches += 1
                    if mismatches > max_mismatches:
                        break

            if mismatches <= max_mismatches:
               occurrences.append(i)

        return occurrences, num_index_hits

In [19]:
from kmer_index import SubseqIndex

p = 'GGCGCGGTGGCTCACGCCTGTAAT'
t = record.seq
k = 8 
ival = 3

subseq_ind = SubseqIndex(t, 8, 3)


occurrences, num_index_hits = query_subseq(p, t, subseq_ind, max_mismatches=2)

print(occurrences)
print(num_index_hits)







[56922, 84641, 147558, 191452, 262042, 273669, 364263, 465647, 635931, 657496, 681737, 717706, 747359]
35


In [20]:
from kmer_index import SubseqIndex

ind = SubseqIndex('ATATAT', 3, 2)
print(ind.index)

[('AAA', 0), ('TTT', 1)]
